In [2]:
import pandas as pd
import numpy as np

In [16]:
records = []
with open("../data/interpro2go") as f:
    for line in f:
        if line.startswith("!"):
            continue
        ipr = line.split(">")[0].split(":")[1].strip().split()[0]
        go = line.strip().split(";")[-1].strip()
        desc1 = line.strip().partition(" ")[2].split(" > ")[0]
        desc2 = line.strip().partition(">")[2].strip().partition(";")[0].strip()
        records.append({"interpro_accession": ipr, "go_term": go, "desc1": desc1, "desc2": desc2})

interpro2go_df = pd.DataFrame(records)
interpro2go_df["go_desc"] = interpro2go_df["desc2"].str.replace("GO:", "")

In [25]:
raw_interpro = pd.read_csv("../data/UP000005640_9606.tsv", sep="\t", usecols = ["protein_accession", "interpro_accession"])
raw_interpro["uniprot_id"] = raw_interpro["protein_accession"].str.split("|").str[1]

raw_interpro = raw_interpro.loc[raw_interpro.interpro_accession != "-", ["uniprot_id", "interpro_accession"]]

In [35]:
final = raw_interpro.merge(
    interpro2go_df[["interpro_accession", "go_desc"]], left_on="interpro_accession", right_on="interpro_accession", how="inner"
).drop(columns=["interpro_accession"]).drop_duplicates()

final.to_json("../data/uniprot_id_to_go.json", orient="records", indent=1)